In [1]:
# Import necessary libraries
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import gd_download  # your module that fetches the CSV from Google Drive
from getYfinanceData import get_stock_info

# Number of stocks: N=1... is literal number, N=0 pre-selected sample, N=-1 is all
N_stocks = -1

PATHFOLDER_INPUT = '../data/d0_repo/'
PATHFOLDER_DOWNLOAD = '../data/d1_ydata/'

#FILE_INPUT = 'Defence.csv'
#popDescription = False
FILE_INPUT = 'PotDat.csv'
popDescription = True

verbose = False

Now is: 2025-04-10 23:16


In [2]:
# Load csv-files in local folder (./input) from Cloud-based repository
gd_download.main(PATHFOLDER_INPUT)

*** Source GoogleDrive-folder til download: repositoryRTBI med id 1eOjCF6kVZrby1j0pe7qeLzADixmHyYlP ***
Download proces:
1. Locating PotDat.csv (1v-4lr9sv0XMUeKcD9gcUMxH16rJz1jyN)
2. File 'PotDat.csv' found and deleted.
3. Downloading new copy of PotDat.csv (100%)
1. Locating Cal.csv (1OR90x9BBoFsVWRo3ZX6SxO2UjYD7FaC9)
2. File 'Cal.csv' found and deleted.
3. Downloading new copy of Cal.csv (100%)
1. Locating Google.csv (1-gKrZy8k4NsiWWHEjjvK6LHwz8WKty49)
2. File 'Google.csv' found and deleted.
3. Downloading new copy of Google.csv (100%)
1. Locating Stamdata.csv (1iEoUY0hbEPUm6iCe8IehYi885262UFBe)
2. File 'Stamdata.csv' found and deleted.
3. Downloading new copy of Stamdata.csv (100%)
1. Locating PotNdxGrp.csv (1liS0dRmcFtpG5kw8iI3zCndkavl2RSr_)
2. File 'PotNdxGrp.csv' found and deleted.
3. Downloading new copy of PotNdxGrp.csv (100%)
1. Locating PotNdx.csv (11-s6F22b9_ijECA2caizoadCRJza16SZ)
2. File 'PotNdx.csv' found and deleted.
3. Downloading new copy of PotNdx.csv (100%)
1. Locati

In [3]:
# Define the file path (assuming the CSV is placed in ./app/input/)
file_path = PATHFOLDER_INPUT+FILE_INPUT

# Read the CSV (semicolon-separated; decimal as dot)
df = pd.read_csv(file_path, sep=';')

# The first column header contains the creation datetime.
creation_datetime = df.columns[0]
print(f"PotDat creation datetime: {creation_datetime}")

# Rename that first column header to 'Yahoo'
df.rename(columns={creation_datetime: 'Yahoo'}, inplace=True)

# Display the first few rows
df.head()

PotDat creation datetime: 10-04-25 23:08


,Yahoo,1869,1868,1867,1866,1865,1864,1863,1862,1861,...,1297,1296,1295,1294,1293,1292,1291,1290,1289,1288
0,^AEX,819.24,796.45,823.89,801.26,841.29,877.42,901.52,905.24,898.80,...,748.04,741.90,738.83,733.03,736.18,724.26,714.32,717.51,707.63,701.16
1,^BTC,79855.33,83124.49,76820.59,78916.28,83849.05,81983.74,86800.06,84977.02,82745.27,...,19922.90,18847.70,17936.70,17438.70,17177.90,16949.30,16823.60,16844.40,16669.40,16675.80
2,^DJAFK,474.66,463.12,452.30,449.19,461.51,493.41,508.43,513.94,506.83,...,495.03,489.03,479.06,480.12,478.15,468.83,463.23,467.80,461.47,458.96
3,^FCHI,7126.02,6863.02,7100.42,6927.12,7274.95,7598.98,7858.83,7876.36,7790.71,...,7023.50,6975.68,6924.19,6869.14,6907.36,6860.95,6761.50,6776.43,6623.89,6594.57
4,^FTSE,7913.25,7679.48,7910.53,7702.08,8054.98,8474.74,8608.48,8634.80,8582.81,...,7844.07,7794.04,7724.98,7694.49,7724.94,7699.49,7633.45,7585.19,7554.09,7451.74


In [4]:
# Filter the "Yahoo" column to remove indices (those starting with '^')
filtered_codes = df['Yahoo'][~df['Yahoo'].str.startswith('^')]

# For testing, pick the first 5 stock codes or a predefined bunch
if (N_stocks==0 ) :
    selected_codes =['ALC.SW', 'CWIGAKLA.CO', 'CTC-A.TO', 'NOVN.SW', 'CFR.SW', 'ROG.SW']
elif (N_stocks>0 ) : 
    selected_codes = filtered_codes.tail(N_stocks).tolist()
else :
    selected_codes = filtered_codes.tolist()        # All stocks in Potentials

print(f'\nSignal {N_stocks}: Data on {len(selected_codes)} stocks called from yfinance')

if verbose:
    print('First stocks:')
    print(selected_codes[0:10])
    print('Last stocks:')
    print(selected_codes[-10:])


Signal -1: Data on 1048 stocks called from yfinance


In [5]:
#Prepare date-marking for output filenames
now_datetime = datetime.fromtimestamp(time.time())
formatted_date = now_datetime.strftime("%Y%m%d-%H%M")
#print('Date-marking for output files:', formatted_date)

#Core filename
if FILE_INPUT == 'Defence.csv' :
    filename_core = 'Defence-'+formatted_date
else :
    if (N_stocks == -1) :
        filename_core = 'StockData-'+formatted_date
    else :
        filename_core = 'StockData-test'
print('Signal:', N_stocks, '  Core filename:', filename_core)

Signal: -1   Core filename: StockData-20250410-2317


In [6]:
# Get stock-info for all stocks in list selected_codes

# Timer start
start_time = time.perf_counter()

# Fetch data for each selected stock code
stock_data_list = []
for code in selected_codes:
    print(f"Processing: {code}")
    try:
        stock_data = get_stock_info(code, popDescription)
        stock_data_list.append(stock_data)
    except Exception as e:
        print(f"Error fetching data for {code}: {e}")
    time.sleep(1)  # not overloading yfinance server

# Create a new DataFrame from the fetched data
new_df = pd.DataFrame(stock_data_list)

# Optional: Reorder columns so that the stock code (e.g., 'Symbol') is the first column
if 'Symbol' in new_df.columns:
    cols = new_df.columns.tolist()
    cols.remove('Symbol')
    new_df = new_df[['Symbol'] + cols]

# About 31 min for all Potentials
end_time = time.perf_counter()
print(f"\nGet_Stock_info's execution time: {(end_time - start_time)/60:.2f} min ----------")

if verbose:
    print(new_df.head())

Processing: 0270.HK
Processing: 0285.HK
Processing: 0388.HK
Processing: 0522.HK
Processing: 0636.HK
Processing: 0700.HK
Processing: 0868.HK
Processing: 0883.HK
Processing: 0939.HK
Processing: 0968.HK
Processing: 0981.HK
Processing: 1072.HK
Processing: 1299.HK
Processing: 1368.HK
Processing: 1398.HK
Processing: 1772.HK
Processing: 1810.HK
Processing: 1833.HK
Processing: 1910.HK
Processing: 1913.HK
Processing: 1919.HK
Processing: 1COV.DE
Processing: 2020.HK
Processing: 2269.HK
Processing: 2318.HK
Processing: 2331.HK
Processing: 2333.HK
Processing: 2359.HK
Processing: 3323.HK
Processing: 3690.HK
Processing: 3988.HK
Processing: 6865.HK
Processing: 6869.HK
Processing: 8035.T
Processing: 9922.HK
Processing: 9926.HK
Processing: ABBV
Processing: ABI.BR
Processing: ABN.AS
Processing: ABNB
Processing: ABR
Processing: ABR-PF
Processing: ABT
Processing: ACA.PA
Processing: ACGL
Processing: ACIW
Processing: ACN
Processing: ACRE
Processing: ACS.MC
Processing: AD.AS
Processing: ADC
Processing: ADEN.SW

In [7]:
from pathlib import Path
def saveCsvFile(path, filename_with_no_extension, df_input, myDelimiter=';', myDecimal='.') :
    ''' 
    Save file to csv-file (index are dropped all along)
       shared.saveCsvFile('./csvAll','Outliers_stockprices_zscores', df_input)
    '''
    dir_path = Path(path)
    filename = f'{filename_with_no_extension}.csv'
    csv_file_path = dir_path / filename  # Path object for the file
    
    # Create the directory if it does not exist
    dir_path.mkdir(parents=True, exist_ok=True)
               
    try:
        # Save the DataFrame to a CSV file with semicolon as separator
        # index droppes (if to be saved it must be done before calling saveCsvFile)
        if 'index' in df_input.columns:
            df_to_save = df_input.reset_index(drop=True)
        else:
            # Use the DataFrame as is
            df_to_save = df_input


        # Save the DataFrame to CSV with index=False to avoid duplicating the index
        df_to_save.to_csv(csv_file_path, sep=myDelimiter, decimal=myDecimal, encoding='utf-8', index=False)
        print(f"\n{filename} shaped {df_to_save.shape} saved in {dir_path}")
    except PermissionError:
        print(f"Permission denied. Unable to save {filename}")
    except Exception as e:
        print(f"When saving {filename} an error occurred:", e)

saveCsvFile(PATHFOLDER_DOWNLOAD, filename_core, new_df)


StockData-20250410-2317.csv shaped (1048, 32) saved in ..\data\d1_ydata


In [8]:
import pyarrow
dir_path = PATHFOLDER_DOWNLOAD
filename = filename_core+'.parquet'
new_df.to_parquet(dir_path+filename)
print(f"\n{filename} shaped {new_df.shape} saved in {dir_path}")


StockData-20250410-2317.parquet shaped (1048, 32) saved in ../data/d1_ydata/
